# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates step-by-step how to load, inspect, and explore the FAIRˆ² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all data entities using their `@id` fields for full reproducibility and schema traceability.

### Dataset Source
Croissant schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Install mlcroissant (if not already installed)
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset's Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display basic metadata: name and description
meta = dataset.metadata  # single object, not a dict/list
print(f"{meta.name}: {meta.description}")

## 2. Data Overview

Review available record sets and their fields, referencing their `@id` fields.

In [ ]:
# List all record sets and their fields using @id
print("Available record sets:\n---------------------")
record_sets = dataset.record_sets()
for rs in record_sets:
    print(f"- Record Set Name: {rs.name}\n  @id: {rs.id}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    - Field Name: {f.name}, @id: {f.id}")
    print()

## 3. Data Extraction

Load data from each record set into separate pandas DataFrames. Reference all record sets and fields by their `@id`.

In [ ]:
# Build a list of all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets()]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

for rs_id in dataframes:
    print(f"Columns in record set '{rs_id}': {list(dataframes[rs_id].columns)}")
    display(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's select a record set relevant for EDA. We'll pick the main clinical data record set (typically the one with the most rows and clinical variables). We'll use `@id` fields to select columns for numeric analysis, normalization, and grouping.

*If you find that columns have long `@id` names, that's expected: referencing by `@id` ensures full disambiguation.*

In [ ]:
# Choose the main clinical record set (choose the first with data)
main_rs_id = None
for rs_id, df in dataframes.items():
    if len(df) > 0:
        main_rs_id = rs_id
        break
if main_rs_id is None:
    raise ValueError("No non-empty record sets found!")

df = dataframes[main_rs_id]
print(f"Using record set: {main_rs_id}, with shape {df.shape}")

# Display column names to select relevant fields (@id)
print("Available columns (@id):\n", list(df.columns))

# Try to find a numeric field automatically for demonstration

numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
# Else, try to coerce an integer-like column
if numeric_field_id is None:
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        except Exception:
            pass

if numeric_field_id is None:
    raise ValueError("No numeric field could be identified in main record set.")
print(f"Numeric field chosen (by @id): {numeric_field_id}")

# Filter records for which the field is greater than its median (for demo)
median_value = df[numeric_field_id].median()
filtered_df = df[df[numeric_field_id] > median_value].copy()
print(f"Filtered records with {numeric_field_id} > median ({median_value}): {len(filtered_df)} rows")

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)

# Show
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to find a categorical/group field to group by (heuristic: first non-numeric column)
group_field_id = None
for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]):
        group_field_id = col
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped filtered data by '{group_field_id}':")
    display(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field, and (if available) a relation by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(6, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping is available, boxplot of numeric field by group
if group_field_id:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

We have successfully loaded the FAIRˆ² dataset using `mlcroissant`, referenced all fields and record sets by their `@id`, and performed basic exploration, filtering, normalization, and visualization on available data. For rigorous downstream analyses, always consult field definitions and the Croissant schema documentation.